# RI #1: Free-form regulatory questions and local-embedding RAGAS

Notebook 04 builds question sets for any corpus. This companion works through
an actual **public-domain regulatory source**, then evaluates saved generated
answers separately from retrieval ranking. It adds the RAGAS comparison the
retrieval-only notebooks do not supply.

By the end, you can check source provenance, distinguish a gold supporting quote
from retrieved evidence, inspect correctness and grounding failures, and read
RAGAS faithfulness, answer relevancy, and semantic similarity on their own terms.

## 1. Source, cohort, and the two execution flags

The source is the official **2024 U.S. Code, 5 U.S.C. §552 (FOIA)** snapshot.
Its URL, HTML digest, extracted-text digest, and attribution are shipped with
the source. This fixed edition is a reproducibility fixture, not current-law
advice. No private contracts or PolicyQA materials are inputs.

The saved study has **six model-generated free-form questions and one explicit
off-topic null control**. Each answerable question has a supporting quotation
that resolves to the original text. This validates the quoted source, not the
quality of the synthetic reference; human review is still needed. The generator
and answer judges in the saved run use the same 35B model, so shared errors
remain possible. New live runs use the shared model roles.

`RUN_EXPENSIVE=False` reads the source and saved outcomes and reconstructs
evaluation contexts without a key or model. The separate `RUN_RAGAS=False`
keeps optional RAGAS execution off even if answering is enabled. Fresh model
work uses the shared model-role settings with reasoning disabled and the shared
Vector credential resolver. Recorded results retain their original model identities.

In [1]:
import json
import sys
from pathlib import Path

from IPython.display import Markdown
from IPython.display import display as render_display


# Replay is the first-run path. Fresh measurements require an explicit opt-in.
RUN_EXPENSIVE = False

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from within the advanced-RAG checkout.")
RI = REPO / "implementations" / "regulatory_policy_qa"
sys.path.insert(0, str(REPO / "aieng-advanced-rag"))
sys.path.insert(0, str(RI))
from advanced_lab.common import saved  # noqa: E402 - use this source checkout


def show_table(headers, rows):
    """Display a compact table without requiring pandas or a model runtime."""

    def line(values):
        return "| " + " | ".join(str(value).replace("|", r"\|").replace("\n", " ") for value in values) + " |"

    render_display(Markdown("\n".join([line(headers), line(["---"] * len(headers)), *(line(row) for row in rows)])))


print("Execution mode:", "fresh experiment enabled" if RUN_EXPENSIVE else "saved reports + local exercises")

Execution mode: saved reports + local exercises


## 2. Audit the source and question-generation method

The original generator wrote a question, a reference answer, and a verbatim
quotation from an excerpt. A source check rejects changes in wording; it allows
whitespace differences only. Six hand-picked source excerpts and one easy null
control make this an executable example, not an independent evaluation set.

For a larger regulatory set, use notebook 04's current statement-first generator,
round-trip checking, near-miss nulls, and held-out documents on a suitable collection
of public texts. Preserve a human-reviewed holdout for final comparisons. The
scores here belong to the earlier six-question fixture and are not attributed
retroactively to that newer generator.

In [2]:
from advanced_lab.regulatory_inputs import input_fingerprints, regulatory_inputs  # noqa: E402 - method exercise cell


RUN_RAGAS = False
source = json.loads((RI / "outputs/advanced/regulatory_source.json").read_text())
print("Source:", source["source_url"])
print("Edition:", source["edition"], "Document:", source["document_id"])
print("Attribution:", source["attribution"])
print("Source text SHA-256:", source["text_sha256"])
if RUN_EXPENSIVE:
    from advanced_lab.answering import measure_regulatory

    report = measure_regulatory()
else:
    report = saved("regulatory")
show_table(
    ["Question ID", "Question", "Gold null?"],
    [(row["query_id"], row["question"], row["is_null"]) for row in report["outcomes"]],
)

Source: https://www.govinfo.gov/content/pkg/USCODE-2024-title5/html/USCODE-2024-title5-partI-chap5-subchapII-sec552.htm
Edition: 2024 Document: 5usc552-2024
Attribution: United States Code, 2024 Edition, U.S. Government Publishing Office; U.S. federal government text, public domain in the United States
Source text SHA-256: 0c27b98b52c737b366645eaece6978098a15be13bd5ba634f181825c0f276cad


| Question ID | Question | Gold null? |
| --- | --- | --- |
| foia-0 | Under what specific condition is a person prohibited from being required to resort to or adversely affected by a matter that was required to be published in the Federal Register but was not published? | False |
| foia-1 | What specific procedural requirements must an agency fulfill when deleting identifying details from records to prevent a clearly unwarranted invasion of personal privacy? | False |
| foia-2 | Under what two specific conditions may an agency rely on, use, or cite an interpretation, staff manual, or instruction against a party other than an agency? | False |
| foia-3 | What specific types of charges are agencies permitted to limit to reasonable standard fees when records are requested for commercial use? | False |
| foia-4 | Under what specific conditions must documents be furnished without charge or at a reduced fee according to clause (iii)? | False |
| foia-5 | Under what specific conditions is an agency permitted to require advance payment of fees from a requester? | False |
| foia-null | What is the published telephone number of the lunar mining permit office? | True |

## 3. Inspect correctness, grounding, and a null decision

The pipeline retrieves five passages with local BGE embeddings and gives their
source previews to the generator. Neither the reference nor the gold support ID
is used in retrieval or answering. Correctness is judged against the reference;
grounding is judged separately against the passages actually returned.

Change `example_index` to inspect an outcome. A correctly located supporting quote
does not guarantee that retrieval found it, that the answer used it, or that the
judge was right. The saved study answers **5/6 answerables correctly**. The single
null control is too easy and too small to establish robust null detection.

In [3]:
example_index = 0
example = report["outcomes"][example_index]
print("Question:", example["question"])
print("Reference:", example["reference"])
print("Gold supporting quotation:", example["quote"])
print("Saved answer:", example["answer"])
print("Retrieved IDs:", example["retrieved_ids"])
print(
    "Abstained:",
    example["abstained"],
    "Correct:",
    example["judgement"]["correct"],
    "Grounded:",
    example["judgement"]["grounded"],
)
print("Correctness reason:", example["judgement"]["reason"])
print("Grounding reason:", example["judgement"]["grounding_reason"])
print("Saved null metrics:", report["summary"]["null_detection"])

Question: Under what specific condition is a person prohibited from being required to resort to or adversely affected by a matter that was required to be published in the Federal Register but was not published?
Reference: A person cannot be adversely affected by an unpublished Federal Register matter unless they have already received actual and timely notice of its terms.
Gold supporting quotation: Except to the extent that a person has actual and timely notice of the terms thereof, a person may not in any manner be required to resort to, or be adversely affected by, a matter required to be published in the Federal Register and not so published.
Saved answer: Except to the extent that a person has actual and timely notice of the terms thereof
Retrieved IDs: ['5usc552-2024::0001', '5usc552-2024::0032', '5usc552-2024::0005', '5usc552-2024::0027', '5usc552-2024::0025']
Abstained: False Correct: True Grounded: True
Correctness reason: The candidate correctly identifies the exception regard

## 4. Reconstruct evaluation inputs from public artifacts

RAGAS needs the question, predicted answer, reference, and contexts read by the
generator. These rows now come from the **shipped public source and saved ordered
retrieved IDs**, with the same 1,200-character passage limit. They need no private
`Arm Store`, new generation, corpus download, or API key. A digest mismatch or an
unknown evidence ID fails before evaluation.

Changing a gold reference affects evaluation, not the saved prediction. To compare
metrics fairly, keep these input rows fixed. Fresh answering variants are stored
locally; the RAGAS cell below evaluates the shipped fixture, not an unlabelled
mixture of fresh answers and old scores.

In [4]:
evaluation_rows = regulatory_inputs()
answerable_rows = [row for row in evaluation_rows if not row["is_null"]]
print(
    "Answerable evaluation rows:",
    len(answerable_rows),
    "Separate null controls:",
    len(evaluation_rows) - len(answerable_rows),
)
print("Input fingerprints:", input_fingerprints())
print("First row keys:", sorted(answerable_rows[0]))
print("First row context lengths:", [len(text) for text in answerable_rows[0]["retrieved_contexts"]])

Answerable evaluation rows: 6 Separate null controls: 1
Input fingerprints: {'regulatory_source.json': 'db973f07d077a0276c3fe388055f6fdde523f5b51dde54e9f916bf068a8ade00', 'regulatory.json': '139a479768f1453619c4aef425bdc4ac358aa632a1d1174aadd7aaffd8171554'}
First row keys: ['is_null', 'query_id', 'reference', 'response', 'retrieved_contexts', 'user_input']
First row context lengths: [1200, 1200, 1200, 1200, 1200]


## 5. Compare the RAGAS definitions

| Metric | Inputs and interpretation | What it does not establish |
| --- | --- | --- |
| Faithfulness | Whether the response's claims are supported by retrieved context | Whether that context or answer is the right one for the task. |
| Answer relevancy | Whether the response addresses the question, using generated questions and local embeddings | Reference correctness or a useful abstention policy. |
| Semantic similarity | Local embedding similarity between response and reference | Exact factual correctness or source grounding. |

RAGAS 0.4.1 uses the cached Vector chat adapter and local
`BAAI/bge-small-en-v1.5` CPU embeddings. No hosted embedding client is constructed.
RAGAS and the harness judges use different protocols, so disagreement is worth
inspecting rather than averaging into a single accuracy score. The null control
is excluded from these three metrics and remains in the separate abstention report.

`RUN_RAGAS=True` needs RAGAS and LangChain from this RI's `evaluation` extra, which the
workspace does not install, and makes judging calls. Install them from the repository root, then
restart the kernel:
`uv pip install --python .venv/bin/python -e "implementations/regulatory_policy_qa[evaluation]"`.
Without them the cell prints that command and stops. The original run used 18 chat calls. Cache
hits can reduce fresh calls; repeated identical prompts reuse a response and are
not independent samples. Exceptions and nonfinite scores fail the run.

In [5]:
from importlib.util import find_spec  # noqa: E402 - optional dependency check


# RAGAS and LangChain come from this RI's own `evaluation` extra, which the
# workspace does not install, so the live path checks for them first.
RAGAS_INSTALL = 'uv pip install --python .venv/bin/python -e "implementations/regulatory_policy_qa[evaluation]"'
ragas_report = None
if RUN_RAGAS:
    missing = [name for name in ("ragas", "langchain") if find_spec(name) is None]
    if missing:
        print(
            f"RUN_RAGAS needs {' and '.join(missing)}: run `{RAGAS_INSTALL}` from the repository root, "
            "then restart the kernel."
        )
    else:
        from advanced_lab.ragas_eval import measure_ragas

        ragas_report = measure_ragas()
else:
    ragas_report = saved("ragas")
if ragas_report is not None:
    show_table(
        ["Metric", "Mean on six answerables"], [(name, f"{mean:.4f}") for name, mean in ragas_report["means"].items()]
    )
    show_table(
        ["Question", "Faithfulness", "Answer relevancy", "Semantic similarity"],
        [
            (
                row["query_id"],
                f"{metric['faithfulness']:.4f}",
                f"{metric['answer_relevancy']:.4f}",
                f"{metric['semantic_similarity']:.4f}",
            )
            for row, metric in zip(answerable_rows, ragas_report["metrics"], strict=True)
        ],
    )
    print("Hosted embedding calls:", ragas_report["hosted_embedding_calls"])
    print("Recorded evaluation usage:", ragas_report["usage"])

| Metric | Mean on six answerables |
| --- | --- |
| faithfulness | 0.6667 |
| answer_relevancy | 0.5409 |
| semantic_similarity | 0.7529 |

| Question | Faithfulness | Answer relevancy | Semantic similarity |
| --- | --- | --- | --- |
| foia-0 | 1.0000 | 0.0000 | 0.7974 |
| foia-1 | 1.0000 | 0.9112 | 0.8149 |
| foia-2 | 1.0000 | 0.9763 | 0.8640 |
| foia-3 | 1.0000 | 0.4384 | 0.6307 |
| foia-4 | 0.0000 | 0.9192 | 0.8697 |
| foia-5 | 0.0000 | 0.0000 | 0.5404 |

Hosted embedding calls: 0
Recorded evaluation usage: {'calls': 18, 'prompt_tokens': 17618, 'completion_tokens': 2818, 'total_tokens': 20436, 'cache_hits': 12, 'empty_responses': 0, 'call_seconds': 303.8, 'by_phase': {'unassigned': {'calls': 18, 'prompt_tokens': 17618, 'completion_tokens': 2818, 'total_tokens': 20436, 'cache_hits': 12, 'empty_responses': 0, 'call_seconds': 303.8}}}


## 6. Design the next regulatory experiment

Choose a policy section with a known source version and provenance. First have a
domain reader check generated questions, references, and near-miss nulls. Keep
question IDs and answer predictions fixed when comparing RAGAS with the harness;
inspect disagreements and calibrate judgments on a human-reviewed subset. When
changing retrieval or answering, rerun generation and judge the resulting answers
on a held-out set, preserving both the control and variant.

Six synthetic questions, same-model judging, and one off-topic null are the limits
of this example. None of the displayed means is an independent benchmark accuracy.
For the next step, adapt the shared tool/agent boundary in
[the starter notebook](99_starter_agent.ipynb). [ANSWERING_EXPERIMENTS.md](ANSWERING_EXPERIMENTS.md)
records models, cached provenance, optional dependencies, and the full-split deferral.